# <font color="#418FDE" size="6.5" uppercase>**Fair vergleichen**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Vergleichen Modelle fair mit identischen Splits, Baselines und Metriken. 
- Speichern, laden und prüfen Modelle aus verschiedenen Frameworks reproduzierbar. 
- Untersuchen Teilgruppenmetriken, Datenverschiebung und Modellkarten für verantwortungsvollen Einsatz. 


## **1. Fairer Modellvergleich**

### **1.1. Identischer Split**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_01_01.jpg?v=1784820989" width="250">



>* Gleiche Splits verhindern zufällige Vorteile
>* Modelle werden dadurch belastbar vergleichbar

>* Gleiche Splits verhindern verzerrte Modellvergleiche
>* Vorverarbeitung nur mit Trainingsdaten lernen

>* Datenstruktur beim Split unbedingt beachten
>* Realistische Splits machen Vergleiche vertrauenswürdig



In [ ]:
#@title Python-Code - Identischer Split

# Wir vergleichen Modelle mit demselben Datensplit.
# Identische Testdaten machen Ergebnisse fair vergleichbar.
# Die Ausgabe zeigt faire und unfaire Genauigkeiten.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Wir laden einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Eine einfache Prüfung verhindert unklare Datenprobleme.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Dieser Split wird für beide Modelle wiederverwendet.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Beide Modelle sehen exakt dieselben Trainingsdaten.
linear_model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)

linear_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)

# Die faire Bewertung nutzt denselben Testdatensatz.
linear_fair = accuracy_score(y_test, linear_model.predict(X_test))
tree_fair = accuracy_score(y_test, tree_model.predict(X_test))

# Ein anderer Split kann den Vergleich verzerren.
X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=7
)

bad_tree = DecisionTreeClassifier(max_depth=4, random_state=42)
bad_tree.fit(X_train_bad, y_train_bad)
tree_unfair = accuracy_score(y_test_bad, bad_tree.predict(X_test_bad))

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Fairer Split: Logistic Regression = {linear_fair:.3f}")
print(f"Fairer Split: Decision Tree = {tree_fair:.3f}")
print(f"Unfairer Vergleich: Decision Tree auf anderem Split = {tree_unfair:.3f}")
print("Merksatz: Vergleiche Modelle nur auf identischen Testbeispielen.")



### **1.2. Frameworks fair vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_01_02.jpg?v=1784820991" width="250">



>* Gleiche Aufgabe, Daten und Metriken festlegen
>* Versteckte Vorteile zwischen Frameworks vermeiden

>* Vorverarbeitung transparent und einheitlich halten
>* Hyperparameter mit vergleichbarem Aufwand optimieren

>* Ergebnisse reproduzierbar und mehrfach prüfen
>* Stabilität, Transparenz und Einsatzfähigkeit bewerten



In [ ]:
#@title Python-Code - Frameworks fair vergleichen

# Dieses Beispiel vergleicht Modelle unter gleichen Bedingungen.
# Identische Splits und Metriken machen Ergebnisse fairer.
# Die Ausgabe zeigt faire und unfaire Genauigkeiten.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir laden einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Eine einfache Prüfung verhindert unklare Fehlermeldungen später.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Beide Modelle erhalten exakt denselben Trainings-Test-Split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Die Baseline zeigt, was ohne echte Mustererkennung möglich ist.
baseline = DummyClassifier(strategy="most_frequent", random_state=42)
baseline.fit(X_train, y_train)
baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test))

# Das Vergleichsmodell nutzt dieselben Daten und dieselbe Metrik.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)
model.fit(X_train, y_train)
fair_accuracy = accuracy_score(y_test, model.predict(X_test))

# Ein anderer Split wirkt wie ein versteckter Vorteil.
X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=7
)

# Dieses Ergebnis ist nicht direkt fair vergleichbar.
model.fit(X_train_bad, y_train_bad)
unfair_accuracy = accuracy_score(y_test_bad, model.predict(X_test_bad))

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Baseline, gleicher Split: {baseline_accuracy:.3f}")
print(f"Logistische Regression, gleicher Split: {fair_accuracy:.3f}")
print(f"Logistische Regression, anderer Split: {unfair_accuracy:.3f}")
print("Merksatz: Fair vergleichen heißt gleiche Daten, Baseline und Metrik.")



### **1.3. Zeit und Speicher**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_01_03.jpg?v=1784820993" width="250">



>* Genauigkeit allein reicht nicht für Modellvergleiche
>* Laufzeit und Speicher früh mitplanen

>* Gleiche Bedingungen für aussagekräftige Messungen
>* Training, Vorhersage und Speicher getrennt bewerten

>* Messungen mehrfach und transparent durchführen
>* Leistung und Ressourcen gemeinsam abwägen



In [ ]:
#@title Python-Code - Zeit und Speicher

# Wir vergleichen Modelle auch nach Ressourcen.
# Gleiche Daten machen Messwerte fairer.
# Die Tabelle zeigt Genauigkeit, Zeit und Speicher.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Ein fester Split hält den Vergleich fair.
data = load_breast_cancer()
X = data.data
Y = data.target

# Diese Prüfung macht die Datengröße bewusst.
if X.shape[0] != len(Y):
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Beide Modelle erhalten exakt dieselben Trainingsdaten.
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.25, stratify=Y, random_state=42
)

# Eine einfache Baseline setzt den Mindestvergleich.
baseline = DummyClassifier(strategy="most_frequent")
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=500, random_state=42)
)

# Wir messen Zeit mit NumPy, ohne Zusatzimporte.
results = []
for name, estimator in [("Baseline", baseline), ("Logistische Regression", model)]:
    start_train = np.datetime64("now", "ns")
    estimator.fit(X_train, y_train)
    end_train = np.datetime64("now", "ns")

    start_predict = np.datetime64("now", "ns")
    predictions = estimator.predict(X_test)
    end_predict = np.datetime64("now", "ns")

    train_ms = (end_train - start_train) / np.timedelta64(1, "ms")
    predict_ms = (end_predict - start_predict) / np.timedelta64(1, "ms")
    accuracy = accuracy_score(y_test, predictions)

    if name == "Baseline":
        memory_kb = 0.1
    else:
        fitted_model = estimator.named_steps["logisticregression"]
        memory_kb = fitted_model.coef_.nbytes + fitted_model.intercept_.nbytes
        memory_kb = memory_kb / 1024

    results.append([name, accuracy, train_ms, predict_ms, memory_kb])

# Eine kleine Tabelle zeigt Leistung und Ressourcen gemeinsam.
columns = ["Modell", "Genauigkeit", "Training ms", "Vorhersage ms", "Speicher KB"]
summary = pd.DataFrame(results, columns=columns)
summary["Genauigkeit"] = summary["Genauigkeit"].round(3)
summary["Training ms"] = summary["Training ms"].round(2)
summary["Vorhersage ms"] = summary["Vorhersage ms"].round(2)
summary["Speicher KB"] = summary["Speicher KB"].round(2)

print("scikit-learn Version:", sklearn.__version__)
print("Fairer Split: gleiche Trainings- und Testdaten für beide Modelle.")
print(summary.to_string(index=False))

# Das Diagramm betont den Kompromiss zwischen Qualität und Zeit.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(summary["Training ms"], summary["Genauigkeit"], s=120)
for row in summary.itertuples(index=False):
    ax.annotate(row.Modell, (row._2, row.Genauigkeit), xytext=(6, 4), textcoords="offset points")
ax.set_title("Fairer Vergleich: Genauigkeit gegen Trainingszeit")
ax.set_xlabel("Trainingszeit in Millisekunden")
ax.set_ylabel("Genauigkeit auf demselben Testsplit")
ax.grid(True, alpha=0.3)
plt.show()



## **2. Reproduzierbare Modelle**

### **2.1. Komplexität fair vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_02_01.jpg?v=1784820983" width="250">



>* Leistung immer mit Aufwand und Umgebung vergleichen
>* Geladene Modelle auf gleiche Ergebnisse prüfen

>* Frameworks brauchen unterschiedliche gespeicherte Zusatzartefakte
>* Zuverlässiges Laden zählt neben Genauigkeit

>* Einfachere Modelle können produktiv besser sein
>* Metadaten sichern faire, reproduzierbare Vergleiche



In [ ]:
#@title Python-Code - Komplexität fair vergleichen

# Wir vergleichen Modellkomplexität unter gleichen Bedingungen.
# Speichern und Laden werden hier simuliert.
# Gleiche Vorhersagen zeigen reproduzierbare Modellartefakte.

import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Wir nutzen einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data

y = data.target
if X.shape[0] != len(y):
    raise ValueError("Daten und Zielwerte passen nicht zusammen.")

# Beide Modelle erhalten exakt denselben fairen Split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Ein einfaches Modell enthält Skalierung und Klassifikator gemeinsam.
simple_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Ein komplexeres Modell nutzt mehr mögliche Entscheidungsregeln.
complex_model = DecisionTreeClassifier(
    max_depth=None, min_samples_leaf=1, random_state=42
)

# Beide Modelle werden unter denselben Trainingsdaten angepasst.
simple_model.fit(X_train, y_train)
complex_model.fit(X_train, y_train)

# Wir prüfen Leistung und eine einfache Komplexitätskennzahl.
simple_accuracy = accuracy_score(y_test, simple_model.predict(X_test))
complex_accuracy = accuracy_score(y_test, complex_model.predict(X_test))

simple_complexity = simple_model.named_steps["logisticregression"].coef_.size
complex_complexity = complex_model.tree_.node_count

# Statt Dateien nutzen wir gespeicherte Referenzvorhersagen im Speicher.
simple_saved_predictions = simple_model.predict(X_test)
complex_saved_predictions = complex_model.predict(X_test)

# Nach dem erneuten Prüfen müssen die Vorhersagen identisch bleiben.
simple_reloaded_predictions = simple_model.predict(X_test)
complex_reloaded_predictions = complex_model.predict(X_test)

simple_reproducible = np.array_equal(
    simple_saved_predictions, simple_reloaded_predictions
)
complex_reproducible = np.array_equal(
    complex_saved_predictions, complex_reloaded_predictions
)

print(f"scikit-learn-Version: {sklearn.__version__}")
print("Fairer Split: identische Trainings- und Testdaten für beide Modelle")
print(f"Logistische Regression: Genauigkeit {simple_accuracy:.3f}, Parameter {simple_complexity}")
print(f"Entscheidungsbaum: Genauigkeit {complex_accuracy:.3f}, Knoten {complex_complexity}")
print(f"Reproduzierbar nach Prüfung: einfach={simple_reproducible}, komplex={complex_reproducible}")



### **2.2. Versionen sicher speichern**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_02_02.jpg?v=1784820985" width="250">



>* Modellversionen mit Entstehungskontext speichern
>* Reproduzierbarkeit und Auswahl nachvollziehbar begründen

>* Modellversionen mit stabilen Metadaten kennzeichnen
>* Umgebung und Zusatzdateien vollständig sichern

>* Archivierte Versionen ermöglichen Audits und Fehleranalysen
>* Modelle als dokumentierte, unveränderbare Artefakte speichern



In [ ]:
#@title Python-Code - Versionen sicher speichern

# Dieses Beispiel speichert Modellversionen nachvollziehbar.
# Metadaten machen Training und Prüfung reproduzierbar.
# Der Vergleich zeigt stabile geladene Vorhersagen.

import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir nutzen einen kleinen, eingebauten Klassifikationsdatensatz.
iris = load_iris()
X = iris.data
y = iris.target

# Eine einfache Prüfung schützt vor unerwarteten Datenformen.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Der feste Split gehört zur gespeicherten Modellversion.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Vorverarbeitung und Modell werden gemeinsam versioniert.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=200, random_state=42)
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

# Statt Dateien verwenden wir hier ein In-Memory-Artefakt.
model_version = {
    "model": model,
    "metadata": {
        "experiment_id": "iris-logreg-v001",
        "data_version": "sklearn-load-iris",
        "split": "test_size=0.25, stratify=y, random_state=42",
        "metric": round(float(accuracy), 3),
        "sklearn_version": sklearn.__version__,
    },
}

# Laden bedeutet hier: dieselbe gespeicherte Version wiederverwenden.
loaded_model = model_version["model"]
loaded_predictions = loaded_model.predict(X_test)

same_predictions = np.array_equal(predictions, loaded_predictions)
metadata = model_version["metadata"]

print("scikit-learn-Version:", metadata["sklearn_version"])
print("Experiment:", metadata["experiment_id"])
print("Datenversion:", metadata["data_version"])
print("Gespeicherte Genauigkeit:", metadata["metric"])
print("Geladene Vorhersagen identisch:", same_predictions)



### **2.3. Modelle sicher laden**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_02_03.jpg?v=1784820987" width="250">



>* Modellumgebung vor Vorhersagen sorgfältig prüfen
>* Vorverarbeitung und Feature-Reihenfolge müssen passen

>* Nur geprüfte Modellquellen verwenden
>* Geladene Modelle isoliert mit Testfällen prüfen

>* Vorhersagen mit Referenzdaten fachlich prüfen
>* Gesamte Pipeline vor Einsatz validieren



In [ ]:
#@title Python-Code - Modelle sicher laden

# Dieses Beispiel prüft ein geladenes Modell sicher.
# Metadaten schützen vor falschen Eingabespalten.
# Referenzvorhersagen bestätigen reproduzierbares Laden.

import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir nutzen einen kleinen, eingebauten Klassifikationsdatensatz.
iris = load_iris()
X = iris.data
y = iris.target

# Eine einfache Prüfung verhindert unerwartete Datenformen.
if X.shape[1] != len(iris.feature_names):
    raise ValueError("Die Feature-Anzahl passt nicht zu den Namen.")

# Der Split ist festgelegt und für Klassen fair geschichtet.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Die Pipeline speichert Vorverarbeitung und Modell gemeinsam.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=200, random_state=42)
)

# Das Modell wird einmal trainiert und danach als Artefakt betrachtet.
model.fit(X_train, y_train)
reference_predictions = model.predict(X_test[:5])
reference_accuracy = accuracy_score(y_test, model.predict(X_test))

# Metadaten beschreiben, was beim Laden erwartet wird.
model_card = {
    "framework": "scikit-learn",
    "sklearn_version": sklearn.__version__,
    "feature_names": list(iris.feature_names),
    "target_name": "iris species",
    "reference_predictions": reference_predictions.copy(),
    "reference_accuracy": round(reference_accuracy, 3),
}

# In diesem Beispiel simuliert diese Variable das geladene Modell.
loaded_model = model
loaded_card = model_card

# Vorhersagen sind nur sinnvoll, wenn die Spaltenreihenfolge stimmt.
current_feature_names = list(iris.feature_names)
if current_feature_names != loaded_card["feature_names"]:
    raise ValueError("Die Feature-Reihenfolge passt nicht zum Modell.")

# Ein Referenztest erkennt Änderungen nach dem Laden.
loaded_predictions = loaded_model.predict(X_test[:5])
if not np.array_equal(loaded_predictions, loaded_card["reference_predictions"]):
    raise ValueError("Die Referenzvorhersagen stimmen nicht überein.")

# Auch die dokumentierte Metrik wird erneut geprüft.
loaded_accuracy = accuracy_score(y_test, loaded_model.predict(X_test))
accuracy_difference = abs(loaded_accuracy - loaded_card["reference_accuracy"])

print(f"scikit-learn-Version: {sklearn.__version__}")
print("Sicherheitsprüfung: Metadaten und Referenzfälle bestanden.")
print(f"Dokumentierte Accuracy: {loaded_card['reference_accuracy']:.3f}")
print(f"Neu berechnete Accuracy: {loaded_accuracy:.3f}")
print(f"Abweichung: {accuracy_difference:.3f}")



## **3. Verantwortungsvoll prüfen**

### **3.1. Inferenz sicher ausführen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_03_01.jpg?v=1784820977" width="250">



>* Eingaben wie im Training vorbereiten
>* Formate, Einheiten und Wertebereiche prüfen

>* Unsicherheit und Ausreißer bewusst erkennen
>* Sensible Vorhersagen menschlich prüfen lassen

>* Versionen und Eingaben nachvollziehbar dokumentieren
>* Betrieb auf Drift und Fairness überwachen



In [ ]:
#@title Python-Code - Inferenz sicher ausführen

# Dieses Beispiel prüft Inferenz vor der Vorhersage.
# Es zeigt sichere Eingaben und Datenverschiebung.
# Am Ende sehen wir klare Prüfentscheidungen.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir nutzen einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer(as_frame=True)
feature_names = ["mean radius", "mean texture", "mean smoothness"]
X = data.frame[feature_names].copy()
y = data.target.copy()

# Der Split bleibt reproduzierbar und fair vergleichbar.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Die Pipeline speichert Vorverarbeitung und Modell gemeinsam.
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
model.fit(X_train, y_train)

# Trainingsbereiche dienen als einfache Sicherheitsgrenzen.
train_min = X_train.min()
train_max = X_train.max()
expected_columns = list(X_train.columns)

# Diese Funktion prüft Schema und Werte vor Inferenz.
def safe_predict(input_frame):
    if list(input_frame.columns) != expected_columns:
        return "abgelehnt: falsche Merkmalsreihenfolge oder Spalten"
    if input_frame.isna().any().any():
        return "abgelehnt: fehlende Werte entdeckt"
    outside = (input_frame < train_min) | (input_frame > train_max)
    if outside.any().any():
        return "abgelehnt: Werte außerhalb des Trainingsbereichs"
    prediction = model.predict(input_frame)[0]
    probability = model.predict_proba(input_frame)[0].max()
    return f"akzeptiert: Klasse {prediction}, Sicherheit {probability:.2f}"

# Wir vergleichen normale und riskante Inferenzfälle.
normal_case = X_test.iloc[[0]].copy()
shifted_case = normal_case.copy()
shifted_case["mean radius"] = shifted_case["mean radius"] * 12

# Eine einfache Teilgruppenmetrik zeigt verantwortungsvolle Kontrolle.
median_texture = X_test["mean texture"].median()
low_group = X_test["mean texture"] <= median_texture
high_group = X_test["mean texture"] > median_texture
predictions = model.predict(X_test)

low_accuracy = accuracy_score(y_test[low_group], predictions[low_group])
high_accuracy = accuracy_score(y_test[high_group], predictions[high_group])
overall_accuracy = accuracy_score(y_test, predictions)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Gesamtgenauigkeit: {overall_accuracy:.2f}")
print(f"Teilgruppe niedrige Textur: {low_accuracy:.2f}")
print(f"Teilgruppe hohe Textur: {high_accuracy:.2f}")
print("Normale Eingabe: " + safe_predict(normal_case))
print("Verschobene Eingabe: " + safe_predict(shifted_case))



### **3.2. Teilgruppen fair bewerten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_03_02.jpg?v=1784820979" width="250">



>* Durchschnittswerte können Gruppenprobleme verdecken
>* Relevante Teilgruppen und Fehlerfolgen prüfen

>* Teilgruppen kontextbezogen und statistisch vorsichtig wählen
>* Mehrere Metriken nach Fehlerfolgen bewerten

>* Ursachen prüfen und Modell gezielt anpassen
>* Ergebnisse dokumentieren und Einsatzgrenzen klar benennen



In [ ]:
#@title Python-Code - Teilgruppen fair bewerten

# Wir prüfen Modellleistung getrennt nach relevanten Teilgruppen.
# Gleiche Testdaten zeigen Unterschiede hinter dem Durchschnitt.
# Die Ausgabe vergleicht Genauigkeit und Fehlerraten.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Ein fester Zufallsgenerator macht die Daten reproduzierbar.
rng = np.random.default_rng(42)

# Wir erzeugen ein kleines Klassifikationsproblem mit zwei Gruppen.
features, target = make_classification(
    n_samples=800, n_features=4, n_informative=3, n_redundant=0,
    class_sep=0.9, random_state=42
)

group = np.where(features[:, 0] > np.median(features[:, 0]), "Gruppe A", "Gruppe B")

# Für Gruppe B machen wir die Aufgabe absichtlich etwas schwieriger.
flip_mask = (group == "Gruppe B") & (rng.random(len(target)) < 0.18)
target = target.copy()
target[flip_mask] = 1 - target[flip_mask]

# Der Split bleibt für Modell und Teilgruppenbewertung identisch.
X_train, X_test, y_train, y_test, group_train, group_test = train_test_split(
    features, target, group, test_size=0.3, stratify=target, random_state=42
)

if len(np.unique(group_test)) != 2:
    raise ValueError("Die Testdaten müssen beide Gruppen enthalten.")

# Ein einfaches Modell reicht für die Fairnessprüfung aus.
model = LogisticRegression(max_iter=300, random_state=42)
model.fit(X_train, y_train)

# Wir berechnen Gesamtwerte und danach dieselben Metriken je Gruppe.
predicted = model.predict(X_test)
overall_accuracy = accuracy_score(y_test, predicted)

rows = []
for group_name in sorted(np.unique(group_test)):
    mask = group_test == group_name
    group_y = y_test[mask]
    group_pred = predicted[mask]
    false_positive = np.mean((group_pred == 1) & (group_y == 0))
    false_negative = np.mean((group_pred == 0) & (group_y == 1))
    rows.append([group_name, mask.sum(), accuracy_score(group_y, group_pred), false_positive, false_negative])

result = pd.DataFrame(
    rows, columns=["Gruppe", "Anzahl", "Genauigkeit", "Falsch positiv", "Falsch negativ"]
)

# Kurze Ausgabe: Durchschnitt plus Teilgruppenvergleich.
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Gesamtgenauigkeit: {overall_accuracy:.2f}")
print(result.round(2).to_string(index=False))



### **3.3. Modellkarte erstellen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_20/Lecture_A/image_03_03.jpg?v=1784820981" width="250">



>* Modellkarten dokumentieren Zweck, Prüfung und Grenzen
>* Sie fördern Transparenz und reproduzierbare Vergleiche

>* Einsatzbereich, Daten und Unterrepräsentation klar dokumentieren
>* Teilgruppenfehler, Datenverschiebung und Unsicherheiten sichtbar machen

>* Sichere Nutzung klar begrenzen und überwachen
>* Nutzen, Grenzen und Risiken verständlich machen



In [ ]:
#@title Python-Code - Modellkarte erstellen

# Diese Modellkarte fasst verantwortungsvolle Prüfungen zusammen.
# Teilgruppenmetriken zeigen mögliche unfaire Leistungsunterschiede.
# Das Ergebnis ist eine kurze Modellkarte.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Wir erzeugen kleine, reproduzierbare Beispieldaten.
features, target = make_classification(
    n_samples=600,
    n_features=5,
    n_informative=3,
    n_redundant=0,
    random_state=42,
)

# Eine künstliche Teilgruppe simuliert unterschiedliche Einsatzbedingungen.
rng = np.random.default_rng(42)
group = np.where(features[:, 0] > 0, "Gruppe A", "Gruppe B")
shifted_feature = features[:, 1] + np.where(group == "Gruppe B", 0.8, 0.0)
features[:, 1] = shifted_feature

# Der Split bleibt für alle Prüfungen identisch.
X_train, X_test, y_train, y_test, group_train, group_test = train_test_split(
    features,
    target,
    group,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Wir trainieren ein einfaches Modell als prüfbares Beispiel.
model = LogisticRegression(max_iter=300, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# Gesamtmetrik und Teilgruppenmetriken werden gemeinsam betrachtet.
overall_accuracy = accuracy_score(y_test, predictions)
rows = []
for group_name in sorted(set(group_test)):
    mask = group_test == group_name
    group_accuracy = accuracy_score(y_test[mask], predictions[mask])
    rows.append([group_name, int(mask.sum()), round(group_accuracy, 3)])

# Eine kleine Tabelle passt gut in eine Modellkarte.
metrics_table = pd.DataFrame(
    rows,
    columns=["Teilgruppe", "Testfälle", "Accuracy"],
)

# Ein einfacher Drift-Hinweis vergleicht Trainings- und Testmittelwerte.
train_mean = float(np.mean(X_train[:, 1]))
test_mean = float(np.mean(X_test[:, 1]))
drift_note = "prüfen" if abs(train_mean - test_mean) > 0.2 else "unauffällig"

# Die Modellkarte verbindet Zweck, Metriken, Grenzen und Nutzung.
print(f"scikit-learn Version: {sklearn.__version__}")
print("Modellkarte: Beispielmodell für binäre Klassifikation")
print("Beabsichtigte Nutzung: Lernbeispiel, keine reale Entscheidung")
print(f"Gesamt-Accuracy: {overall_accuracy:.3f}")
print(metrics_table.to_string(index=False))
print(f"Datenverschiebung Merkmal 2: {drift_note}")
print("Grenze: künstliche Daten ersetzen keine echte Fachprüfung")
print("Empfehlung: vor Einsatz Teilgruppen und Drift erneut prüfen")



# <font color="#418FDE" size="6.5" uppercase>**Fair vergleichen**</font>


In this lecture, you learned to:
- Vergleichen Modelle fair mit identischen Splits, Baselines und Metriken. 
- Speichern, laden und prüfen Modelle aus verschiedenen Frameworks reproduzierbar. 
- Untersuchen Teilgruppenmetriken, Datenverschiebung und Modellkarten für verantwortungsvollen Einsatz. 

<font color='yellow'>Congratulations on completing this course!</font>